# 012 — Multi-LLM Routing with LangGraph
**Generation Series** | Fast model for simple queries, powerful model for complex ones

Route queries intelligently:
- **Simple** (factual, lookup) → `llama-3.1-8b-instant` (fast, cheap)
- **Complex** (reasoning, synthesis, multi-step) → `llama-3.3-70b-versatile`
- **Code generation** → specialized prompt + larger model

LangGraph models the routing as a conditional edge.


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Build shared retriever ───────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c)
        for text in [ai_text, ml_text, dl_text]
        for c in splitter.split_text(text[:15000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✓ Shared retriever over {len(docs)} docs")


✓ Shared retriever over 47 docs


In [5]:
# ── Query classifier chain ───────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Classify the user query into one of three categories:\n"
        "- 'simple': factual lookup, definition, yes/no\n"
        "- 'complex': reasoning, comparison, multi-step synthesis\n"
        "- 'code': code generation or debugging\n"
        "Reply with ONLY the category word."
    )),
    ("human", "Query: {question}"),
])
classifier = classify_prompt | llm | StrOutputParser()

test_queries = [
    "What year was the Turing test proposed?",
    "Compare and contrast supervised, unsupervised, and reinforcement learning.",
    "Write a Python function to tokenize text using NLTK.",
    "What is a neural network?",
]
for q in test_queries:
    cat = classifier.invoke({"question": q}).strip().lower()
    print(f"  [{cat:8}] {q}")


  [simple  ] What year was the Turing test proposed?
  [complex ] Compare and contrast supervised, unsupervised, and reinforcement learning.


  [code    ] Write a Python function to tokenize text using NLTK.
  [simple  ] What is a neural network?


In [6]:
# ── Generation chains per route ───────────────────────────────────────────
simple_prompt = ChatPromptTemplate.from_template(
    "Answer concisely (1-2 sentences).\nContext: {context}\nQ: {question}\nA:"
)
complex_prompt = ChatPromptTemplate.from_template(
    "Provide a thorough, structured answer using the context.\n"
    "Context:\n{context}\nQuestion: {question}\nAnswer:"
)
code_prompt = ChatPromptTemplate.from_template(
    "Write clean, well-commented Python code.\n"
    "Context:\n{context}\nTask: {question}\nCode:"
)

simple_chain  = simple_prompt  | llm       | StrOutputParser()
complex_chain = complex_prompt | llm_smart | StrOutputParser()
code_chain    = code_prompt    | llm_smart | StrOutputParser()
print("✓ Three generation chains ready")


✓ Three generation chains ready


In [7]:
# ── LangGraph routing pipeline ───────────────────────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, END

class RouterState(TypedDict):
    question:  str
    route:     str
    context:   str
    answer:    str

def classify_node(state: RouterState):
    route = classifier.invoke({"question": state["question"]}).strip().lower()
    if route not in ("simple", "complex", "code"):
        route = "complex"
    print(f"  Route: {route}")
    return {**state, "route": route}

def fetch_context(state: RouterState):
    docs = retriever.invoke(state["question"])
    ctx  = "\n\n".join(d.page_content for d in docs)
    return {**state, "context": ctx[:3000]}

def generate_simple(state: RouterState):
    ans = simple_chain.invoke({"context": state["context"], "question": state["question"]})
    return {**state, "answer": ans}

def generate_complex(state: RouterState):
    ans = complex_chain.invoke({"context": state["context"], "question": state["question"]})
    return {**state, "answer": ans}

def generate_code(state: RouterState):
    ans = code_chain.invoke({"context": state["context"], "question": state["question"]})
    return {**state, "answer": ans}

def route_decision(state: RouterState):
    return state["route"]

graph = StateGraph(RouterState)
graph.add_node("classify",  classify_node)
graph.add_node("fetch_ctx",  fetch_context)
graph.add_node("simple",     generate_simple)
graph.add_node("complex",    generate_complex)
graph.add_node("code",       generate_code)
graph.set_entry_point("classify")
graph.add_edge("classify", "fetch_ctx")
graph.add_conditional_edges("fetch_ctx", route_decision,
                             {"simple": "simple", "complex": "complex", "code": "code"})
graph.add_edge("simple",  END)
graph.add_edge("complex", END)
graph.add_edge("code",    END)
router_app = graph.compile()
print("✓ Router pipeline compiled")


✓ Router pipeline compiled


/home/saurabh/miniconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [8]:
# ── Test the router ──────────────────────────────────────────────────────
import time

for q in test_queries:
    t0 = time.time()
    result = router_app.invoke({"question": q, "route": "", "context": "", "answer": ""})
    elapsed = time.time() - t0
    print(f"Q [{result['route']:8}] {q[:55]}")
    print(f"A: {result['answer'][:200]}")
    print(f"   ↳ {elapsed:.2f}s\n")


  Route: simple


Q [simple  ] What year was the Turing test proposed?
A: The Turing test was proposed by Alan Turing in 1950, in his paper "Computing Machinery and Intelligence".
   ↳ 0.44s



  Route: complex


Q [complex ] Compare and contrast supervised, unsupervised, and rein
A: **Introduction to Machine Learning Paradigms**

Machine learning is a field of study that enables machines to learn from data and make predictions or decisions. There are three primary paradigms in ma
   ↳ 2.00s

  Route: code


Q [code    ] Write a Python function to tokenize text using NLTK.
A: ### Tokenization Function using NLTK
#### Overview
This function takes a string of text as input and returns a list of tokens.

#### Code
```python
import nltk
from nltk.tokenize import word_tokenize

   ↳ 1.08s

  Route: simple


Q [simple  ] What is a neural network?
A: A neural network is an artificial system modeled after the human brain, designed to recognize patterns in data through machine perception. It interprets sensory data by translating real-world inputs i
   ↳ 0.36s



## Key Takeaways
- Routing cuts cost 3-5× by using cheap models for simple queries
- The classifier itself should use the fast model (meta-routing would be silly)
- In production: add a cache layer before routing (see **016_semantic_cache.ipynb**)
- LangGraph's conditional edges make routing logic explicit and unit-testable
